# System-Prompt A/B

**What does your system prompt cost you?** Keep one question fixed and run it at one model under two (or more) different system prompts, the standing instructions / persona / guardrails, with your own key. See each answer, its token cost, and latency side by side. The classic lesson: a vague, unbounded system prompt uses more tokens and costs more than a tight, bounded one that gets the same answer. About a minute, free, no signup.

This notebook is **pre-filled with a demonstration default**: a vague system prompt next to a tight one. Run it as-is and watch the vague one cost more for the same answer. The lesson is the token and cost numbers, not a verdict the tool emits.

A free, own-key tool by [pysolvr / LLM Prover](https://llmprover.pysolvr.com).

---

> **Free tool, provided as-is, no warranty.** It uses *your own* API key and calls the provider directly. **Your key never leaves your machine / your Colab account and is never sent to pysolvr.** You are responsible for your own usage and costs. Outputs are informational, not advice. pysolvr is not liable for decisions made from these results.

**Key safety:** this notebook never asks you to type a key into a cell (see step 2). Keys live in Colab Secrets or your shell environment, so a saved or shared copy contains no secrets.

## 1. Setup

Pulls the tool from the repo and installs the provider SDKs. `openai` also covers Gemini, Grok, Together, and Groq via their OpenAI-compatible endpoints.

In [ ]:
!pip -q install openai>=1.0 anthropic>=0.30
!wget -q -O system_prompt_ab_core.py https://raw.githubusercontent.com/dAIvdMercer/llmprover-tools/main/tools/system-prompt-ab/system_prompt_ab_core.py
!wget -q -O prices.json              https://raw.githubusercontent.com/dAIvdMercer/llmprover-tools/main/tools/system-prompt-ab/prices.json
print('ready')

## 2. Add your API key (do NOT type it into a cell)

**Never paste an API key directly into a notebook cell.** A key typed into a cell gets saved into the notebook file, your Colab history, and any copy you share. Store it *outside* the notebook instead. The cell below only *reads* it.

### On Colab (recommended), use the Secrets panel
1. Click the **key icon** (Secrets) in the left sidebar.
2. **Add a secret** for your model's provider, using the exact name:

   | Secret name | Models it unlocks |
   |-------------|-------------------|
   | `OPENAI_API_KEY` | `gpt-*`, `o3`/`o4-*` |
   | `ANTHROPIC_API_KEY` | `claude*` |
   | `GEMINI_API_KEY` | `gemini*` |
   | `XAI_API_KEY` | `grok*` |
   | `TOGETHER_API_KEY` | `<org>/<model>` |
   | `GROQ_API_KEY` | `qwen/*` |

3. Toggle **Notebook access** on for the secret. You only need the ONE provider whose model you use.

### On local Jupyter, environment variable
Set it in your shell *before* launching Jupyter: `export OPENAI_API_KEY=sk-...` then `jupyter lab`.

Then run the cell below. **Nothing is sent to pysolvr; your key stays on your machine / in your Colab account.**

In [ ]:
# Loads keys from Colab Secrets (preferred) or the environment. Contains no key itself.
import os

_KEY_NAMES = ['OPENAI_API_KEY', 'ANTHROPIC_API_KEY', 'GEMINI_API_KEY', 'XAI_API_KEY', 'TOGETHER_API_KEY', 'GROQ_API_KEY']
try:
    from google.colab import userdata  # Colab only
    for _name in _KEY_NAMES:
        try:
            _val = userdata.get(_name)
            if _val:
                os.environ[_name] = _val
        except Exception:
            pass  # secret not set / notebook access off -- skip
except ImportError:
    pass  # not on Colab -- keys come from the shell environment

_loaded = [n for n in _KEY_NAMES if os.environ.get(n)]
print('keys loaded for:', _loaded if _loaded else 'NONE -- add one via the Colab Secrets panel (key icon, left) with Notebook access ON, or set an env var. See cell above.')

## 3. Pick a model, your question, and two system prompts

The cell below is pre-filled with a demonstration: the **same question** (how to reset a password) asked under two **system prompts**, a vague, unbounded one and a tight, bounded one. Run it as-is and compare the token counts, cost, and latency. The vague system prompt tends to produce a longer, costlier answer for the same question.

The *system prompt* is the standing instruction the model follows (persona, tone, format, guardrails). The *user prompt* is the actual question, held constant here so only the system prompt changes.

Use the provider's own model-ID string (the tool is model-agnostic). Examples: `gpt-5`, `claude-opus-5-5`, `gemini-4-argon`. To find what exists and what it costs: **[llmprover.pysolvr.com/models](https://llmprover.pysolvr.com/models/)** and **[llmprover.pysolvr.com/api-pricing](https://llmprover.pysolvr.com/api-pricing/)**.

To test **your own**: set `PROMPT` to your question and `SYSTEMS` to 2 to 5 of your own system prompts.

In [ ]:
MODEL  = 'gpt-4o-mini'
PROMPT = 'How do I reset my password?'   # the user question, held constant

SYSTEMS = [
    'You are a helpful assistant.',
    ('You are a support agent. Answer in at most 3 short numbered steps. No preamble, no sign-off, '
     'no extra explanation.'),
]

# Optional labels, same order as SYSTEMS. Set to None to auto-label 'System 1', 'System 2', ...
LABELS = ['vague / unbounded', 'tight / bounded']

## 4. Run it

The system prompts fire concurrently, each with the same user question. You get each response, token cost (from the dated price snapshot), and latency, side by side, plus a mechanical summary (the heaviest system prompt used Nx the tokens of the lightest and cost Nx as much). **No verdict** -- these are token and cost counts, not a judgment of which system prompt is better.

In [ ]:
from system_prompt_ab_core import system_prompt_ab_core, format_result

result = system_prompt_ab_core(model=MODEL, user_prompt=PROMPT, systems=SYSTEMS, labels=LABELS)
print(format_result(result))

---

**Prices are a dated snapshot, not a live feed.** For the live, authoritative table see [llmprover.pysolvr.com/api-pricing](https://llmprover.pysolvr.com/api-pricing/).

If you're interested in more free tools to test and measure different aspects of an AI pipeline, see our [free AI tools](https://llmprover.pysolvr.com/tools/). For more advanced features, system prompts scored for quality across your workload, the which-layer diagnostic, and drift tracked over time, see [Features](https://llmprover.pysolvr.com/features/?tool=system-prompt-ab).